 Step 1 Business Understanding

 Step 2 Data Collection

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Installing libraries

In [3]:
pip install pandas numpy gensim scikit-learn streamlit nltk fastapi uvicorn joblib python-multipart requests python-dotenv gunicorn spacy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 91.0 MB/s eta 0:00:00


In [5]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Importing libraries

In [5]:
import pandas as pd
import numpy as np
import re
import string
import pickle
import joblib

import nltk
from nltk.corpus import stopwords

import spacy
from gensim.models import Word2Vec

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

from tqdm import tqdm

from fastapi import FastAPI
import uvicorn

In [8]:
legal_clause = pd.read_csv("/content/legal_docs.csv")
legal_clause.head()

,Unnamed: 0,clause_text,clause_type,totalwords,totalletters
0,0,"Make any Investments, except:",investments,4.0,30.0
1,1,No more than 45% of the “value” (as defined i...,investments,76.0,460.0
2,2,"Make or hold any Investments, except:",investments,6.0,38.0
3,3,The SubAdviser is hereby authorized and direc...,investments,228.0,1474.0
4,4,"Make any advance, loan, extension of credit (...",investments,52.0,329.0


In [9]:
legal_clause.shape

(21187, 5)

Step 3 Data Preparation


In [10]:
legal_clause.isnull().sum()

,0
Unnamed: 0,0
clause_text,43
clause_type,0
totalwords,26
totalletters,26


In [11]:
legal_clause.duplicated().sum()

np.int64(0)

In [12]:
legal_clause = legal_clause.dropna(subset=["clause_text"])

In [13]:
legal_clause.isnull().sum()

,0
Unnamed: 0,0
clause_text,0
clause_type,0
totalwords,0
totalletters,0


 Step 4 Data Cleaning

In [14]:
import spacy
import re
import string
import nltk
from nltk.corpus import stopwords

# Download stopwords once
nltk.download("stopwords")

# Load spaCy model once
nlp = spacy.load("en_core_web_sm")

# Load stopwords once
stop_words = set(stopwords.words("english"))

def preprocess_text(text):
    # Handle missing values
    if not isinstance(text, str):
        return []

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Lemmatization
    doc = nlp(text)

    tokens = []

    for token in doc:
        if (
            token.text not in stop_words
            and not token.is_punct
            and len(token.text) > 1
        ):
            tokens.append(token.lemma_)

    return tokens

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Testing

In [15]:
sample = "The Employee @376 shall maintain confidentiality of all company information."

print("Original:")
print(sample)

print("\nProcessed:")
print(preprocess_text(sample))

Original:
The Employee @376 shall maintain confidentiality of all company information.

Processed:
['employee', 'shall', 'maintain', 'confidentiality', 'company', 'information']


Step 5 Model Building

In [16]:
legal_clause['tokens'] = legal_clause["clause_text"].apply(preprocess_text)

In [17]:
tokenized_clauses = legal_clause["tokens"].tolist()

In [18]:
model = Word2Vec(
    sentences=tokenized_clauses,
    vector_size=200,
    window=5,
    min_count=2,
    workers=4)

Step 6 Model Training

In [19]:
model.train(
    tokenized_clauses,
    total_examples=model.corpus_count,
    epochs=20
)

(18205465, 21566180)

In [20]:
len(model.wv)

7631

In [21]:
# Get the vector for one word
model.wv['termination']          # returns a numpy array of shape (vector_size,)

# Get vectors for all words as a matrix
model.wv.vectors                 # shape: (len(model.wv), vector_size)
print(model.wv.vectors.shape)

(7631, 200)


In [22]:
# Confirm every vocab word has a vector
print(len(model.wv))              # number of words
print(model.wv.vectors.shape[0])  # should match — number of vectors
print(model.wv.vectors.shape[1])  # should match vector_size

7631
7631
200


In [23]:
score_1 = model.wv.most_similar("termination")
score_2 = model.wv.similarity("termination","confidential")

In [24]:
print(score_1)
print(score_2)


[('expiration', 0.5144063234329224), ('commencement', 0.45817431807518005), ('terminate', 0.4565597474575043), ('resignation', 0.4369952082633972), ('effective', 0.42821937799453735), ('effectiveness', 0.3932419419288635), ('death', 0.37324854731559753), ('early', 0.355317622423172), ('start', 0.3460422158241272), ('maturity', 0.3376733660697937)]
0.0035240806


Converting clause into a single vector

In [25]:
def clause_to_vector(tokens, model):
    # Store the vector for each word in this clause
    vectors = []

    for word in tokens:
        # Only use words that exist in the trained vocabula ry
        if word in model.wv:
            vectors.append(model.wv[word])

    # If no words are in vocabulary
    if len(vectors) == 0:
        # Return a zero vector so downstream code doesn't break
        # (keeps output shape consistent even for empty/unknown clauses)
        return np.zeros(model.vector_size)

    # Average all word vectors
    # This collapses word-level vectors into a single clause-level vector
    # by averaging across all word vectors, dimension-wise
    return np.mean(vectors, axis=0)

Step 7 Model Testing

In [26]:
def find_similar_clauses(query,model,clause_vectors,df,top_k=5):

    # Step 1: Preprocess query
    tokens = preprocess_text(query)

    # Step 2: Convert query to vector
    query_vector = clause_to_vector(tokens, model)

    # Step 3: Reshape for cosine similarity
    query_vector = query_vector.reshape(1, -1)

    # Step 4: Calculate similarity
    similarities = cosine_similarity(
        query_vector,
        clause_vectors
    )[0]

    # Step 5: Get Top K indices
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Step 6: Prepare results
    results = []

    for idx in top_indices:
        results.append({
            "clause": df.iloc[idx]["clause_text"],
            "similarity_score": round(
                similarities[idx], 4
            )
        })

    return results

Testing

In [27]:
clause_vectors = np.array([clause_to_vector(tokens, model)
                          for tokens in legal_clause["tokens"]])

In [28]:
query = "The employee shall preserve company secrets."

results = find_similar_clauses(
    query=query,
    model=model,
    clause_vectors=clause_vectors,
    df=legal_clause,
    top_k=5
)

for r in results:
    print(r)

{'clause': 'As used in this Agreement, the term "Confidential Information" means any and all of the Company\'s trade secrets, confidential and proprietary information and all other information and data of the Company that is not generally known to third persons who could derive economic value from its use or disclosure, including, without limitation, the Company\'s profile of prospective or current vendors or customers, business methods and structure, details of the Company\'s contracts and business matters, employee compensation, personnel information, marketing strategies and plans, business plans, pricing information and strategies, costs information, and financial data, whether or not reduced to writing or other tangible medium of expression, including work product created by Employee in rendering services to the Company. During his employment with the Company and thereafter, Employee will not use or disclose to others any of the Confidential Information except as authorized in wri

Step 8 - Model Evaluation

In [30]:
def precision_at_k(relevant, retrieved, k):
    retrieved_k = retrieved[:k]

    relevant_count = len(
        set(retrieved_k).intersection(set(relevant))
    )

    return relevant_count / k

Testing

In [31]:
relevant = [1, 2, 5]
retrieved = [1, 2, 3, 4, 5]

precision_at_k(
    relevant,
    retrieved,
    k=5
)

0.6

In [32]:
model.save('word2vec.model')

In [33]:
import pickle

pickle.dump(
    clause_vectors,
    open("clause_vectors.pkl","wb")
)

In [35]:
legal_clause.to_pickle("clauses.pkl")